# 2 — Put different ECGs on an explicit time grid

**Learning objective:** explain anti-aliased sample-rate conversion, build a
fixed-duration window with an honest padding mask, and form a PyTorch batch.
We will distinguish mathematical interpolation from newly measured information.

Our path is **native signal → canonical leads → full-record resampling → window →
right padding if needed → batch**. The final default representation is float32 mV
with shape `(batch, 12, 5000)` at 500 Hz for ten seconds. There are no labels or models.

We use the actual functions in [`signal.py`](../src/ecg_waveform_lab/signal.py):
`resample_signal`, `make_window`, `prepare_ecg` and `collate_windows`. Synthetic sine
waves isolate aliasing; real public examples provide the ECG context.

## Setup: use this checkout's installed package

Run `uv sync --locked` and launch `uv run --frozen jupyter lab notebooks` from the
repository root. Choose its Python 3 kernel and run this notebook in order.
Each tutorial works independently. The installed package locates the bundled
sample relative to the checkout; there are no machine-specific paths or downloads.
The environment versions below make your numerical session inspectable.

In [ ]:
import os
from importlib.metadata import version

from ecg_waveform_lab import project_root

root = project_root()
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from ecg_waveform_lab import (
    CANONICAL_LEADS,
    calibrate,
    collate_windows,
    load_catalog,
    make_window,
    prepare_ecg,
    read_native,
    reorder_leads,
    resample_signal,
    verify_sample,
    WaveformDataset,
)

display(pd.Series({
    package: version(package)
    for package in ("numpy", "scipy", "wfdb", "torch", "matplotlib")
}, name="Installed version"))

**What we learned:** this independent notebook uses the installed package and bundled
public sample. Intermediate arrays remain in memory; the source files are unchanged.

## 2.1 Select a higher-rate example

Choose one 1000 Hz PTB recording to demonstrate conversion to 500 Hz. Its native
recording is longer than ten seconds. We decode and order the leads before changing
the sample grid. Only structural information is used for selection.

In [ ]:
catalog = load_catalog(root)
record = next(item for item in catalog if item.cohort == "ptb")
native = read_native(record.header_path)
ordered_millivolts = reorder_leads(native.millivolts, native.lead_names)
print("Native signal shape:", ordered_millivolts.shape)
print("Native rate (Hz):", native.sample_rate_hz)
print("Native duration (s):", ordered_millivolts.shape[1] / native.sample_rate_hz)

Sampling frequency fixes the time interval, `Δt = 1/fs`. The first ten seconds at
500 Hz contain 5000 samples at times 0 through 9.998 s. There is no sample exactly
at 10 s in that half-open interval. Duration `N/fs` and the last timestamp `(N-1)/fs`
answer different questions. Compare those quantities before resampling.

In [ ]:
target_rate_hz = 500
source_interval_seconds = 1 / native.sample_rate_hz
target_interval_seconds = 1 / target_rate_hz
print("Native sample interval (ms):", 1000 * source_interval_seconds)
print("Target sample interval (ms):", 1000 * target_interval_seconds)
print("Ten-second target sample count:", 10 * target_rate_hz)
print("Last timestamp in that window (s):", (10 * target_rate_hz - 1) / target_rate_hz)

**What we learned:** changing sample rate changes the time grid, not the physical
meaning of mV. A fixed sample count without a stated rate does not define a duration.

## 2.2 Resample with an anti-aliasing filter

When reducing the rate, frequencies above the new Nyquist limit (`fs/2`) would fold
into lower frequencies. An anti-aliasing low-pass filter attenuates those components
before decimation. Simply keeping every second sample does not do this.

The package uses SciPy's `resample_poly`: a rational ratio `up/down`, a low-pass FIR
filter with a Kaiser window (parameter 5.0), and efficient polyphase evaluation.
Conceptually this is upsample → filter → downsample; the implementation avoids
explicitly computing all inserted-zero samples. For 1000 → 500 Hz the ratio is 1/2;
257 → 500 Hz needs 500/257. No additional notch or ECG-specific bandpass is applied.

We resample the **whole recording before cropping**, retaining real context around
future window boundaries. At the true recording ends, linear extension is an
explicit assumption, not measured data. Very long records use more memory.

In [ ]:
from fractions import Fraction

conversion_ratio = (
    Fraction(str(target_rate_hz)) / Fraction(str(native.sample_rate_hz))
)
print("Upsampling factor:", conversion_ratio.numerator)
print("Downsampling factor:", conversion_ratio.denominator)
resampled_millivolts = resample_signal(
    ordered_millivolts, native.sample_rate_hz, target_rate_hz
)
print("Resampled shape:", resampled_millivolts.shape)
print("Resampled nominal duration (s):", resampled_millivolts.shape[1] / target_rate_hz)

Compare the same lead on the two time grids. **Try changing** `target_rate_hz` to
250 or 1000 and rerun from the time-grid section. With 250 Hz, the sample count
falls further and the Nyquist limit is lower. At the original rate the package
returns a copy. A smoother curve does not imply more diagnostic information.

In [ ]:
lead_name = "II"
view_seconds = 2.0
channel = CANONICAL_LEADS.index(lead_name)
native_count = round(view_seconds * native.sample_rate_hz)
output_count = round(view_seconds * target_rate_hz)
native_time = np.arange(native_count) / native.sample_rate_hz
output_time = np.arange(output_count) / target_rate_hz

figure, axis = plt.subplots(figsize=(11, 4), constrained_layout=True)
axis.plot(native_time, ordered_millivolts[channel, :native_count], label="Native")
axis.plot(
    output_time, resampled_millivolts[channel, :output_count],
    linestyle="--", label=f"Resampled to {target_rate_hz} Hz",
)
axis.set(
    xlabel="Time (s)",
    ylabel="Voltage (mV)",
    title=f"Same real ECG, lead {lead_name}",
)
axis.legend()
axis.grid(alpha=0.2)
plt.show()

**What we learned:** the actual package controls the conversion and keeps units
unchanged. Filter boundary assumptions and reduced bandwidth matter scientifically.

## 2.3 Make aliasing visible with a synthetic signal

An ECG mixes many frequencies, so use a clearly labelled synthetic sum of a 10 Hz
sine and a 350 Hz sine, sampled at 1000 Hz. When reducing to 500 Hz, the new Nyquist
limit is 250 Hz. Naive decimation makes 350 Hz look like 150 Hz; a low-pass resampler
should suppress most of the 350 Hz component.

Try `high_frequency_hz = 200.0`: this component is below the new Nyquist limit and
should remain. This experiment uses the production resampler, not a replacement.

In [ ]:
high_frequency_hz = 350.0
synthetic_time = np.arange(2000) / 1000
low_component = np.sin(2 * np.pi * 10 * synthetic_time)
high_component = 0.7 * np.sin(2 * np.pi * high_frequency_hz * synthetic_time)
synthetic_mixture = low_component + high_component
synthetic_channels = np.tile(synthetic_mixture, (12, 1))

filtered_channels = resample_signal(synthetic_channels, 1000, 500)
naively_decimated = synthetic_mixture[::2]
expected_low_component = low_component[::2]
comparison_time = np.arange(filtered_channels.shape[1]) / 500
print("Synthetic input shape:", synthetic_channels.shape)
print("Filtered shape:", filtered_channels.shape)

Plot an interior interval to reduce edge effects. The naive trace retains the
folded component; the filtered trace is close to the low-frequency component when
350 Hz is selected. This does not mean a generic resampler removes all ECG artefact.

In [ ]:
visible = (comparison_time >= 0.2) & (comparison_time < 0.5)
figure, axis = plt.subplots(figsize=(11, 4), constrained_layout=True)
axis.plot(
    comparison_time[visible],
    naively_decimated[visible],
    label="Naive decimation",
    alpha=0.65,
)
axis.plot(
    comparison_time[visible],
    filtered_channels[0,
    visible],
    label="Polyphase FIR result",
)
axis.plot(
    comparison_time[visible], expected_low_component[visible],
    linestyle="--", label="10 Hz component alone",
)
axis.set(
    xlabel="Time (s)",
    ylabel="Synthetic amplitude (mV)",
    title="Aliasing experiment; not a clinical ECG",
)
axis.legend()
axis.grid(alpha=0.2)
plt.show()

**What we learned:** anti-alias filtering prevents high-frequency content from
masquerading as a lower-frequency signal. Upsampling likewise interpolates; it
cannot recover information absent from the measurements.

## 2.4 Select a window only after resampling

A window has an explicit start and duration. Try `start_seconds = 5.0` and inspect
a different part of the same long recording. Times must align to the output sample
grid; this API rejects a fractional-sample start rather than silently rounding.
The operation does not claim that this window represents an entire recording.

In [ ]:
start_seconds = 0.0
window_seconds = 10.0
window = make_window(
    resampled_millivolts,
    target_rate_hz,
    duration_seconds=window_seconds,
    start_seconds=start_seconds,
    short_policy="reject",
    record_id=record.record_id,
)
print("Window tensor:", tuple(window.waveform.shape), window.waveform.dtype)
print("Start (s):", window.start_seconds)
print("Supported time samples:", int(window.valid_time_mask.sum()))
print("Transformation flags:", window.flags)

**What we learned:** resampling and windowing are distinct operations. A 500 Hz,
ten-second window has 5000 samples; a 250 Hz version has 2500. Selecting a window
can omit events elsewhere, independent of whether the tensor has the right shape.

## 2.5 Represent a short recording honestly

The sample includes a real nine-second CPSC recording. At 500 Hz a ten-second
representation needs an extra second. `prepare_ecg` right-pads with 0 mV and returns
a boolean mask marking which target-grid samples are supported by the observed
recording rather than appended padding. Interpolated samples are still derived
values, not extra measured information. The mask is not a clinical quality judgement.

In [ ]:
short_record = next(item for item in catalog if item.duration_seconds < 10)
short_native = read_native(short_record.header_path)
short_window = prepare_ecg(short_native, sample_rate_hz=500)
observed_samples = int(short_window.valid_time_mask.sum())
print("Original duration (s):", short_record.duration_seconds)
print("Prepared shape:", tuple(short_window.waveform.shape))
print("Observed-duration samples:", observed_samples)
print("Padded samples:", int((~short_window.valid_time_mask).sum()))
print("Flags:", short_window.flags)

Shade the padded region explicitly. The flat tail was introduced by the program;
it must not be mistaken for an acquired period of electrical silence. Choosing
`short_policy="reject"` instead gives an explicit error and no substitute signal.

In [ ]:
padded_time = np.arange(short_window.waveform.shape[1]) / 500
padding_start_seconds = observed_samples / 500
figure, axis = plt.subplots(figsize=(11, 4), constrained_layout=True)
axis.plot(padded_time, short_window.waveform[1].numpy(), label="Lead II tensor")
axis.axvspan(
    padding_start_seconds,
    10,
    color="orange",
    alpha=0.25,
    label="Synthetic zero padding",
)
axis.set(
    xlabel="Time (s)",
    ylabel="Voltage (mV)",
    title="Real short ECG with an explicit padding boundary",
)
axis.legend()
axis.grid(alpha=0.2)
plt.show()

**What we learned:** padding produces a regular shape while changing the meaning
of part of the array. A padding mask does not automatically make convolutions,
pooling or later calculations insensitive to those zeros.

## 2.6 Stack windows and broadcast the time mask

Batch construction introduces a leading example axis. To compare compatible
objects, prepare the long and short examples at the same explicit 500 Hz setting,
regardless of earlier rate experiments. The collator preserves record identities,
units, canonical leads and masks, and refuses incompatible windows.

In [ ]:
long_window = prepare_ecg(native, sample_rate_hz=500)
batch = collate_windows([long_window, short_window])
print("Waveform batch:", tuple(batch["waveform"].shape))
print("Time masks:", tuple(batch["valid_time_mask"].shape))
print("Record order:", batch["record_ids"])
print("Units:", batch["units"])

A `(batch, time)` mask becomes `(batch, 1, time)` so it can broadcast over all
channels. The observed-only mean sums voltage over time and divides by the number
of supported samples for each recording. This is an arithmetic demonstration,
not a new preprocessing normalisation or a fit to the sample population.

In [ ]:
values = batch["waveform"]
time_mask = batch["valid_time_mask"]
channel_broadcast_mask = time_mask[:, None, :]
voltage_sums = (values * channel_broadcast_mask).sum(dim=-1)
observed_counts = time_mask.sum(dim=-1)[:, None]
observed_means = voltage_sums / observed_counts
print("Broadcast mask:", tuple(channel_broadcast_mask.shape))
print("Channel sums:", tuple(voltage_sums.shape))
print("Denominator:", tuple(observed_counts.shape))
print("Observed-only means:", tuple(observed_means.shape))

ECG means can be close to zero, hiding the denominator effect. In a synthetic
vector `[1, 3, 0, 0]` with two padded positions, the mean of observed values is 2;
the mean including padding is 1. Try another pair of nonzero observed values.

In [ ]:
illustrative_values = torch.tensor([1.0, 3.0, 0.0, 0.0])
illustrative_mask = torch.tensor([True, True, False, False])
print("Mean including padding:", illustrative_values.mean().item())
print("Observed-only mean:", illustrative_values[illustrative_mask].mean().item())

**What we learned:** each axis has a role: batch identifies examples, channels
identify leads, and time follows a declared grid. A mask describes padding and
must be handled deliberately in each later calculation.

**Explain back:** why filter before decimating? Why resample before cropping?
Why does upsampling not create new measured information? Why is a time mask
insufficient on its own to make a later model padding-invariant?

Continue to [validation and the Dataset interface](03_real_record_validation.ipynb).